# Patient MDM — Notebook complet de traitement des données
### De l'audit des sources à la réconciliation contrôlée

**Objectif :** documenter et rendre reproductible le pipeline du projet Patient Master Data Management à partir du jeu de données synthétique inclus dans le dossier.

**Scénarios :** `S1_5pct`, `S2_15pct`, `S3_30pct`.

**Étapes couvertes**
1. Initialisation et repérage des données.
2. Chargement des sources et audit structurel.
3. Profilage de la qualité des données.
4. Exploration des données standardisées.
5. Analyse des paires candidates et des décisions de matching.
6. Exploration des Golden Records, memberships et lineage.
7. Inspection des files de revue humaine et du journal d'audit.
8. Préparation d'une réconciliation contrôlée.
9. Synthèse des résultats et limites.

> **Séparation des responsabilités :** `ground_truth.csv`, `hard_nonmatch_evaluation.csv` et `reference_population_private.csv` sont des fichiers réservés à l'évaluation. Ils ne sont pas chargés dans les étapes de matching de ce notebook. Le notebook lit les résultats de matching déjà calculés; il ne réentraîne ni ne remplace silencieusement le moteur de matching.

## 1. Initialisation

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 100)

SCENARIOS = ["S1_5pct", "S2_15pct", "S3_30pct"]

# Recherche le dossier de données à partir du répertoire du notebook.
candidates = [
    Path.cwd() / "patient_mdm_synthetic_dataset",
    Path.cwd().parent / "patient_mdm_synthetic_dataset",
    Path("/mnt/data/Patient_MDM_Dashboard/patient_mdm_synthetic_dataset"),
]
DATA_DIR = next((p.resolve() for p in candidates if p.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Dossier patient_mdm_synthetic_dataset introuvable. "
        "Placez le notebook dans le dossier Patient_MDM_Dashboard, à côté du dossier de données."
    )
print("Dossier de données :", DATA_DIR)
print("Scénarios trouvés :", [p.name for p in DATA_DIR.iterdir() if p.is_dir() and p.name.startswith("S")])

## 2. Inventaire des fichiers

In [ ]:
inventory = []
for path in sorted(DATA_DIR.rglob("*")):
    if path.is_file():
        inventory.append({
            "fichier": str(path.relative_to(DATA_DIR)),
            "extension": path.suffix.lower(),
            "taille_Ko": round(path.stat().st_size / 1024, 2)
        })
inventory_df = pd.DataFrame(inventory)
display(inventory_df.head(40))
print(f"Nombre total de fichiers : {len(inventory_df):,}")

## 3. Audit structurel des données brutes

On vérifie la présence des fichiers de chaque système, le nombre de lignes, les colonnes, les identifiants manquants ou dupliqués et la répartition des identités par source. Cette étape décrit les données; elle ne corrige pas les fichiers originaux.

In [ ]:
IDENTITY_FILES = [
    "admission_patients.csv",
    "lab_patients.csv",
    "radiology_patients.csv",
    "consultation_patients.csv",
    "billing_patients.csv",
]

def read_csv_safe(path):
    try:
        return pd.read_csv(path)
    except (FileNotFoundError, pd.errors.EmptyDataError):
        return pd.DataFrame()

raw_audit_rows = []
raw_samples = {}
for scenario_name in SCENARIOS:
    scenario_dir = DATA_DIR / scenario_name
    for filename in IDENTITY_FILES:
        path = scenario_dir / filename
        df = read_csv_safe(path)
        raw_samples[(scenario_name, filename)] = df
        id_col = "source_patient_id" if "source_patient_id" in df.columns else None
        raw_audit_rows.append({
            "scenario": scenario_name,
            "fichier": filename,
            "existe": path.exists(),
            "lignes": len(df),
            "colonnes": len(df.columns),
            "identifiants_manquants": int(df[id_col].isna().sum()) if id_col else None,
            "identifiants_dupliques": int(df[id_col].duplicated().sum()) if id_col else None,
        })
raw_audit = pd.DataFrame(raw_audit_rows)
display(raw_audit)

### Exemple d'une source brute

In [ ]:
example_raw = raw_samples[("S1_5pct", "admission_patients.csv")]
display(example_raw.head())
print("Colonnes :", example_raw.columns.tolist())

## 4. Profilage des champs et des valeurs manquantes

Les valeurs manquantes et anomalies peuvent être intentionnelles dans un jeu de données synthétique conçu pour tester le matching. Le profilage sert à les mesurer, pas à supposer qu'elles sont toutes des erreurs à supprimer.

In [ ]:
profile_rows = []
for scenario_name in SCENARIOS:
    for filename in IDENTITY_FILES:
        df = raw_samples[(scenario_name, filename)]
        if df.empty:
            continue
        for col in df.columns:
            profile_rows.append({
                "scenario": scenario_name,
                "fichier": filename,
                "champ": col,
                "valeurs_manquantes": int(df[col].isna().sum()),
                "taux_manquant_pct": round(float(df[col].isna().mean() * 100), 2),
                "valeurs_distinctes": int(df[col].nunique(dropna=True)),
            })
field_profile = pd.DataFrame(profile_rows)
display(field_profile.query("scenario == 'S1_5pct'").sort_values("taux_manquant_pct", ascending=False).head(25))

In [ ]:
missing_by_scenario = (field_profile.groupby("scenario", as_index=False)
                       .agg(taux_manquant_moyen_pct=("taux_manquant_pct", "mean")))
display(missing_by_scenario)
ax = missing_by_scenario.set_index("scenario")["taux_manquant_moyen_pct"].plot(kind="bar", title="Taux moyen de valeurs manquantes par scénario")
ax.set_ylabel("% moyen par champ")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Chargement des données standardisées

Les fichiers standardisés conservent les champs d'origine et ajoutent des champs `std__...` et `match__...`. Ils sont utilisés comme préparation des entrées pour le matching. Les données brutes restent disponibles pour comparaison et traçabilité.

In [ ]:
def load_standardized(scenario_name):
    folder = DATA_DIR / scenario_name / "standardized"
    frames = []
    for path in sorted(folder.glob("*.csv")):
        df = read_csv_safe(path)
        if not df.empty:
            df["standardized_file"] = path.name
            frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

standardized = {s: load_standardized(s) for s in SCENARIOS}
std_summary = []
for s, df in standardized.items():
    std_summary.append({
        "scenario": s,
        "enregistrements_standardisés": len(df),
        "systèmes_sources": df["source_system"].nunique() if "source_system" in df else 0,
        "IDs_sources_distincts": df[["source_system", "source_patient_id"]].drop_duplicates().shape[0]
            if {"source_system", "source_patient_id"}.issubset(df.columns) else 0,
        "DOB_non_parsées": int(df["std__dob_parse_status"].astype(str).str.contains("fail|invalid|error", case=False, na=False).sum())
            if "std__dob_parse_status" in df else None,
    })
display(pd.DataFrame(std_summary))
display(standardized["S1_5pct"].head(5))

## 6. Matching baseline — résultats calculés

Cette section lit les paires candidates et les résumés de résultats déjà produits par le pipeline. Elle ne construit pas de nouvelles paires et n'utilise aucune vérité terrain comme entrée du matching.

In [ ]:
matching_summary = read_csv_safe(DATA_DIR / "matching_baseline_summary.csv")
matching_eval = read_csv_safe(DATA_DIR / "matching_baseline_evaluation.csv")
display(matching_summary)
if not matching_eval.empty:
    display(matching_eval)

In [ ]:
pair_counts = []
for s in SCENARIOS:
    path = DATA_DIR / s / "matching_baseline" / "candidate_pairs.csv"
    pairs = read_csv_safe(path)
    if not pairs.empty:
        pair_counts.append({
            "scenario": s,
            "paires_candidates": len(pairs),
            "MATCH_AUTO": int((pairs.get("decision", pd.Series(dtype=str)) == "MATCH_AUTO").sum()),
            "REVIEW": int((pairs.get("decision", pd.Series(dtype=str)) == "REVIEW").sum()),
            "NON_MATCH": int((pairs.get("decision", pd.Series(dtype=str)) == "NON_MATCH").sum()),
        })
pair_counts_df = pd.DataFrame(pair_counts)
display(pair_counts_df)
if not pair_counts_df.empty:
    pair_counts_df.set_index("scenario")[["MATCH_AUTO", "REVIEW", "NON_MATCH"]].plot(
        kind="bar", stacked=True, title="Répartition des décisions de matching"
    )
    plt.ylabel("Nombre de paires")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

### Exemple de paires candidates

Les identifiants locaux de systèmes différents sont conservés. Une paire candidate ne constitue pas à elle seule une fusion : le moteur la classe selon les règles configurées et les cas ambigus sont envoyés en revue humaine.

In [ ]:
pairs_s1_path = DATA_DIR / "S1_5pct" / "matching_baseline" / "candidate_pairs.csv"
pairs_s1 = read_csv_safe(pairs_s1_path)
display(pairs_s1.head(10))
print("Colonnes de la table de paires :", pairs_s1.columns.tolist())

## 7. Golden Records et membership

Un Golden Record représente une identité consolidée. `cluster_membership.csv` permet de retrouver les enregistrements sources associés à chaque Golden Record. Les fichiers lineage et de conflits permettent de justifier les valeurs retenues et les cas nécessitant une revue.

In [ ]:
golden_summary = read_csv_safe(DATA_DIR / "golden_record_summary.csv")
golden_eval = read_csv_safe(DATA_DIR / "golden_record_evaluation.csv")
display(golden_summary)
if not golden_eval.empty:
    display(golden_eval)

In [ ]:
golden_examples = {}
membership_examples = {}
for s in SCENARIOS:
    golden_examples[s] = read_csv_safe(DATA_DIR / s / "golden_record" / "golden_records.csv")
    membership_examples[s] = read_csv_safe(DATA_DIR / s / "golden_record" / "cluster_membership.csv")
    print(f"{s}: {len(golden_examples[s]):,} Golden Records; {len(membership_examples[s]):,} memberships")
display(golden_examples["S1_5pct"].head())
display(membership_examples["S1_5pct"].head(10))

### Examiner les membres d'un Golden Record

In [ ]:
if not golden_examples["S1_5pct"].empty and not membership_examples["S1_5pct"].empty:
    sample_gr_id = str(golden_examples["S1_5pct"].iloc[0]["golden_record_id"])
    print("Golden Record choisi :", sample_gr_id)
    display(golden_examples["S1_5pct"].query("golden_record_id == @sample_gr_id"))
    display(membership_examples["S1_5pct"].query("golden_record_id == @sample_gr_id"))

## 8. Files de revue humaine et audit

Les cas `PENDING` n'ont pas encore de décision humaine. Les statuts `REVIEWED_MATCH`, `REVIEWED_NON_MATCH` et `ESCALATED` reflètent les décisions saisies dans le dashboard. Le journal d'audit doit conserver la trace des actions. Le notebook lit ces fichiers mais ne simule pas de décisions.

In [ ]:
human_queue = read_csv_safe(DATA_DIR / "human_review_queue.csv")
attribute_queue = read_csv_safe(DATA_DIR / "attribute_conflict_review_queue.csv")
audit_log = read_csv_safe(DATA_DIR / "mdm_audit_log.csv")

def status_summary(df, label):
    if df.empty or "current_status" not in df:
        return pd.DataFrame(columns=["type", "statut", "nombre"])
    out = df.groupby(["scenario", "current_status"]).size().reset_index(name="nombre")
    out.insert(0, "type", label)
    return out

display(status_summary(human_queue, "Paires"))
display(status_summary(attribute_queue, "Conflits attributs"))
display(audit_log.tail(10))

## 9. Réconciliation contrôlée — aperçu sans écriture

La fonction ci-dessous prépare une proposition à partir des seules décisions explicitement confirmées par un réviseur. Elle n'utilise pas `ground_truth.csv`, `hard_nonmatch_evaluation.csv` ou `reference_population_private.csv`. Les résultats de la proposition sont affichés sans écraser les Golden Records de référence.

**Important :** cette v1 est conservatrice et illustrative. Elle ne remplace pas une revue métier ni une procédure de production.

In [ ]:
from reconciliation import build_reconciliation

reconciliation_preview = build_reconciliation(DATA_DIR, "S1_5pct")
preview_summary = pd.DataFrame([{
    "scenario": reconciliation_preview["scenario"],
    "Golden Records de référence": reconciliation_preview["baseline_golden_count"],
    "Golden Records proposés": reconciliation_preview["reconciled_golden_count"],
    "décisions humaines confirmées": reconciliation_preview["confirmed_decisions_count"],
    "fusions acceptées": int((reconciliation_preview["accepted_decisions"].get("result", pd.Series(dtype=str)) == "MERGED").sum()),
    "fusions refusées": len(reconciliation_preview["rejected_decisions"]),
}])
display(preview_summary)
display(reconciliation_preview["accepted_decisions"].head(20))
display(reconciliation_preview["rejected_decisions"].head(20))

### Écrire une version réconciliée (optionnel)

La cellule suivante est volontairement désactivée par défaut. Décommentez-la uniquement si vous voulez créer les fichiers `reconciliation_v1/` pour le scénario choisi. Les fichiers de référence ne sont pas remplacés.

In [ ]:
# ATTENTION : action d'écriture sur disque, optionnelle.
# from reconciliation import save_reconciliation
# output_dir = save_reconciliation(reconciliation_preview)
# print("Version réconciliée créée dans :", output_dir)

## 10. Synthèse automatique par scénario

In [ ]:
summary_rows = []
for s in SCENARIOS:
    g = golden_examples.get(s, pd.DataFrame())
    m = membership_examples.get(s, pd.DataFrame())
    p = read_csv_safe(DATA_DIR / s / "matching_baseline" / "candidate_pairs.csv")
    h = human_queue[human_queue["scenario"] == s] if not human_queue.empty and "scenario" in human_queue else pd.DataFrame()
    a = attribute_queue[attribute_queue["scenario"] == s] if not attribute_queue.empty and "scenario" in attribute_queue else pd.DataFrame()
    summary_rows.append({
        "scenario": s,
        "identités_sources_standardisées": len(standardized.get(s, pd.DataFrame())),
        "paires_candidates": len(p),
        "Golden_Records": len(g),
        "membres_cluster": len(m),
        "paires_en_attente": int((h.get("current_status", pd.Series(dtype=str)) == "PENDING").sum()),
        "conflits_attributs_en_attente": int((a.get("current_status", pd.Series(dtype=str)) == "PENDING").sum()),
    })
final_summary = pd.DataFrame(summary_rows)
display(final_summary)

## 11. Conclusions et limites

- Les données sont synthétiques et servent à tester le pipeline MDM, pas à prendre des décisions médicales.
- La standardisation prépare les champs à la comparaison mais ne garantit pas l'identité d'une personne.
- Le matching produit des propositions, avec une file de revue pour les cas ambigus.
- Le Golden Record consolide les attributs et conserve une relation avec les enregistrements sources.
- Les décisions humaines doivent être traçables et les fusions doivent respecter des contrôles de sécurité.
- La réconciliation v1 n'applique pas encore la sélection de valeur des conflits d'attributs à la survivorship.
- Pour un usage de production, il faudra une base de données transactionnelle, des droits d'accès, des tests, une journalisation protégée, une procédure de rollback et une validation de conformité.

### Fichiers d'évaluation à garder séparés
Les fichiers `ground_truth.csv`, `hard_nonmatch_evaluation.csv` et `reference_population_private.csv` ne sont pas utilisés comme entrées du matching dans ce notebook. Ils peuvent être consultés séparément dans une étape d'évaluation clairement isolée, après que les prédictions ont été générées.